## Secure Water Treatment (SWaT) Dataset Dictionary
----
### Datetime
*   Timestamp — Date and time of the recorded data point

### Sensors -> continuous
*   `LIT101` — Level Indicator Transmitter at stage
*   `FIT101` — Flow Indicator Transmitter at stage
*   `FIT201` — Flow Indicator Transmitter at stage 2
*   `FIT301`, `LIT301` — Flow and Level indicators at stage 3
*   `FIT401`, `LIT401` — Flow and Level indicators at stage 4
*   `FIT501`, `FIT502`, `FIT503`, `FIT504` — Flow indicators at stage 5
*   `FIT601` — Flow Indicator Transmitter at stage 6
*   `AIT201`, `AIT202`, `AIT203` — Analyzer Indicators for pH, conductivity, and ORP at stage 2
*   `AIT401`, `AIT402` — Analyzer Indicators at stage 4
*   `AIT501`, `AIT502`, `AIT503`, `AIT504` — Analyzer Indicators at stage 5
*   `DPIT301` — Differential Pressure Indicator Transmitter at stage 3
*   `PIT501`, `PIT502`, `PIT503` — Pressure Indicator Transmitters at stage 5

### Actuators -> discrete
*   `MV101` — Motorized Valve at stage 1
*   `MV201` — Motorized Valve at stage 2
*   `MV301`, `MV302`, `MV303`, `MV304` — Motorized Valves at stage 3
*   `P101`, `P102` — Pumps at stage 1
*   `P201`, `P202`, `P203`, `P204`, `P205`, `P206` — Pumps at stage 2
*   `P301`, `P302` — Pumps at stage 3
*   `P401`, `P402`, `P403`, `P404` — Pumps at stage 4
*   `P501`, `P502` — Pumps at stage 5
*   `P601`, `P602`, `P603` — Pumps at stage 6
*   `UV401` — UV disinfection unit

### Label -> binary
*   `Normal/Attack` — Label indicating whether the data point corresponds to normal or attack operation

In [1]:
import os
import joblib
import random
import seaborn as sns
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px

from statsmodels.tsa.seasonal import seasonal_decompose
from datetime import datetime, timedelta
from numpy.fft import fft, fftfreq

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import acf, pacf

In [ ]:
# Clean column names and date reformatting
def clean_data(
        data, 
        timestamp_col='Timestamp', 
        format='%d/%m/%Y %I:%M:%S %p', 
        target_col='Normal/Attack'):
    
    df = data.copy()

    # Reformat time for convenience
    df[timestamp_col] = df[timestamp_col].str.strip()
    df[timestamp_col] = pd.to_datetime(df[timestamp_col], format=format)

    # Encode target feature
    df[target_col] = (df[target_col]
                      .str.replace(" ","", regex=False)
                      .map({'Normal':0, 'Attack':1}))

    return df


# Fill missing timestamps from the sequence
def fill_missing_timestamps(
        data, actuators, sensors, target,
        timestamp_col='Timestamp', freq='s'):
    
    df = data.copy()

    # Get the missing timestamps within the range of min and max
    start, end = df[timestamp_col].min(), df[timestamp_col].max()
    full_range = pd.date_range(start, end, freq=freq)

    missing_timestamps = full_range[~full_range.isin(df[timestamp_col])]
    print(f"Missing timestamp: {len(missing_timestamps)}")
    print(f"Dataset length: {len(df)}")

    # Use reindex method to automatically expand the missing timestamps with NaN
    df = df.set_index(timestamp_col).reindex(full_range).rename_axis(timestamp_col).reset_index()

    df[actuators] = df[actuators].fillna(0)
    df[sensors] = df[sensors].interpolate(method='linear', limit_direction="both")
    df[target] = df[target].fillna(0)

    print(f"Dataset length after filling: {len(df)}")
    print(f"Missing value remaining: {df.isna().sum().sum()}")

    return df, missing_timestamps

In [ ]:
swat_clean.describe().T

In [ ]:
# Visualize some features
def visualization(data, timestamp, column, target):
    df = data.copy()

    plt.figure(figsize=(10,3))
    plt.plot(df[timestamp], df[column], alpha=0.5, label='normal')
    plt.scatter(df[df[target] == 1][timestamp], df[df[target] == 1][column], color='red', alpha=0.5, label='attacked')

    plt.legend()
    plt.tight_layout()
    plt.show()


# Running example
cols = swat.columns[6:9]
for j in cols:
    visualization(swat_fill, "Timestamp", j, "Normal/Attack")

In [ ]:
# Show density plots for sampled features
def feature_densities(data, columns):
    plt.figure(figsize=(8,4))
    sns.kdeplot(data[columns], fill=True, palette='magma')

    plt.title("Density Plots")
    plt.tight_layout()
    plt.show()


# Running example
feature_densities(swat_fill, ['LIT101', 'LIT301', 'LIT401'])
feature_densities(swat_fill, ['AIT201', 'AIT202', 'AIT203'])

In [ ]:
# Experiment in data smoothing using moving averages
# The ideal implementation would not use smoothing, preventing anomalies from being hidden
def data_smoothing(data, columns, window, samples):
    SMA = data[columns].rolling(window, min_periods=1).mean()       # Smiple Moving Average smoothing
    EMA = data[columns].ewm(span=window, adjust=False).mean()       # Exponential Moving Average smoothing

    weights = np.arange(1, window + 1)                              # Weigthed Moving Average smoothing
    WMA = data[columns].rolling(window, min_periods=1).apply(
        lambda x: np.dot(x, weights[-len(x):]) / weights[-len(x):].sum(),
        raw=True)

    # Take some random sequence
    idx = random.randint(0, len(data) - samples - 1)

    original_sampled = np.array(data.loc[idx:idx + samples, columns])
    sma_sampled = np.array(SMA.loc[idx:idx + samples])
    ema_sampled = np.array(EMA.loc[idx:idx + samples])
    wma_sampled = np.array(WMA.loc[idx:idx + samples])

    plt.figure(figsize=(12, 5))
    plt.plot(original_sampled, color='pink', label='Original')
    plt.plot(sma_sampled, color='orange', label='SMA')
    plt.plot(ema_sampled, color='maroon', label='EMA')
    plt.plot(wma_sampled, color='navy', label='WMA')

    plt.title(f"Data Smoothing - {window} Windows")
    plt.legend()
    plt.grid(linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.show()


# Running example
data_smoothing(
    swat_fill,
    columns='LIT301',
    window=300,
    samples=3600)

In [ ]:
# Check features correlation
def heatmap(data):
    # Calculate the correlation matrix
    matrix = data.corr()

    # Using Plotly for interactive visual
    fig = px.imshow(
        matrix,
        text_auto=False,
        zmin=-1,
        zmax=1,
        labels=dict(color="Correlation"),
        width=800,
        height=800)

    fig.update_layout(
        xaxis_title=None,
        yaxis_title=None,
        yaxis_autorange='reversed')

    fig.update_xaxes(tickangle=90)
    fig.show()


# Runnnig example
heatmap(swat_fill)

In [ ]:
# Inspect time-series decomposition
def decompose(df, timestamp, feature):
    series = df[[timestamp, feature]].set_index(timestamp, drop=True)

    result = seasonal_decompose(series, model='additive', period=3600*6)
    components = {
        'Observed': result.observed,
        'Trend': result.trend,
        'Seasonal': result.seasonal,
        'Residual': result.resid}

    colors = ['#781c6d', '#a52c60', '#cf4446', '#ed6925']
    line_styles = ['-', '--', ':', '-.']

    fig, axes = plt.subplots(4, 1, figsize=(10, 8), sharex=False)
    for ax, (label, data), color, style in zip(axes, components.items(), colors, line_styles):
        ax.plot(data, label=label, color=color, linestyle=style)
        ax.set_title(label, fontsize=12, fontweight='bold')
        ax.grid(axis='y', linestyle='--', alpha=0.6)
        ax.legend()

    plt.tight_layout()
    plt.show()

    return {
        "original": result.observed,
        "trend": result.trend,
        "seasonal": result.seasonal,
        "error": result.resid}


# Running example
decomposition = decompose(
    swat_fill,
    timestamp="Timestamp",
    feature='LIT301')

In [ ]:
# Inspect basic distribution of residuals
# Normal distribution is expected, but real data may often not follow this assumption
def residual_distribution(error):
    plt.figure(figsize=(6,4))
    sns.kdeplot(error, fill=True, color='navy', alpha=0.4)
    plt.title("Error Distribution")

    plt.xlabel("Values")
    plt.ylabel("Density")
    plt.tight_layout()
    plt.show()


# Running example
residual_distribution(decomposition['error'])

In [ ]:
# Check Fourier Transform for frequency analysis
def fourier_transform(df, feature, sampling_rate=1.0):
    series = df[['Timestamp', feature]].set_index('Timestamp', drop=True)
    series_dm = np.array(series).flatten() - np.mean(np.array(series)) # flattened to 1D array

    N = len(series_dm)
    fft_output = fft(series_dm)
    frequencies = fftfreq(N, d=1/sampling_rate)

    # 3. Take positive frequencies (since FFT is symmetric)
    half_N = N // 2
    pos_frequencies = frequencies[:half_N]
    # Normalize magnitude
    magnitude = np.abs(fft_output)[:half_N] * (2.0 / N)

    # 4. Plotting
    fig, axes = plt.subplots(2, 1, figsize=(8, 6))

    # Plot 1: Demeaned Time-Series Signal
    axes[0].plot(series_dm, color='blue')
    axes[0].set_title(f"Demeaned Time Series - {feature}")
    axes[0].set_xlabel("Time Steps")
    axes[0].set_ylabel("Amplitude")
    axes[0].grid(True)

    # Plot 2: FFT Frequency Spectrum
    axes[1].semilogy(pos_frequencies, magnitude, color='red')
    axes[1].set_title("Fourier Transform (Frequency Spectrum)")
    axes[1].set_yscale('log')
    axes[1].set_xlabel("Frequency")
    axes[1].set_ylabel("Magnitude")
    axes[1].grid(True)

    plt.tight_layout()
    plt.show()


# Running example
fourier_transform(
    swat_fill,
    feature='LIT301',
    sampling_rate=1.0)